HomeGuard Security System Simulator

Author: Louise Plessis

Description: A smart home monitoring system that processes sensor readings and triggers alerts for security, safety, and comfort issues.

In [25]:
import random
from datetime import datetime
 
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

In [26]:
# Function to create a sensor & alerts

def create_sensor(sensor_id, location, sensor_type, threshold=None):
    sensor = {
        "sensor_id": sensor_id,
        "location": location,
        "sensor_type": sensor_type,
        "threshold": threshold
    }
    return sensor
 
def create_alert(severity, message, sensor_id, timestamp):
    alert = {
        "severity": severity,
        "message": message,
        "sensor_id": sensor_id,
        "timestamp": timestamp

    }
    return alert
 
# Initialize sensors for the Peterson home
sensors = [
    create_sensor("s1", "Living Room", "motion"),
    create_sensor("s2", "Kitchen", "temperature", 30),
    create_sensor("s3", "Front Door", "door"),
    create_sensor("s4", "Bedroom", "smoke")
]

print(sensors[0])
print(sensors[1])
print(sensors[2])
print(sensors[3])

{'sensor_id': 's1', 'location': 'Living Room', 'sensor_type': 'motion', 'threshold': None}
{'sensor_id': 's2', 'location': 'Kitchen', 'sensor_type': 'temperature', 'threshold': 30}
{'sensor_id': 's3', 'location': 'Front Door', 'sensor_type': 'door', 'threshold': None}
{'sensor_id': 's4', 'location': 'Bedroom', 'sensor_type': 'smoke', 'threshold': None}


In [27]:
# Conditional logic to check for abnormal readings based on system mode and sensor thresholds 

def is_abnormal_reading(sensor, reading_value):
    sensor_type = sensor["sensor_type"]

    if sensor_type == "temperature":
        return reading_value < 35 or reading_value > 95
    elif sensor_type == "motion":
        return reading_value == True
    elif sensor_type == "door":
        return reading_value == "OPEN"
    elif sensor_type == "smoke":
        return reading_value == "DETECTED"
    else:
        return False

 
 
def should_trigger_security_alert(sensor, reading_value, system_mode):
    # Only consider triggering if the reading is abnormal in the first place
    if not is_abnormal_reading(sensor, reading_value):
        return False

    sensor_type = sensor["sensor_type"]

    # Smoke & temperature are always a security alert, whatever the mode
    if sensor_type == "smoke" or sensor_type == "temperature":
        return True

    # Motion and door depend on whether anyone should be moving around
    if sensor_type in ("motion", "door"):
        if system_mode == "AWAY" or system_mode == "SLEEP":
            return True
        else:
            return False

    return False   # <- fallback: any other case, don't alert
        


In [28]:
# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False
 
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False

34°F is abnormal: True
68°F is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


In [29]:
# Building function

def generate_reading(sensor):
    sensor_type = sensor["sensor_type"]

    if sensor_type == "temperature":
        return random.randint(30, 100)          # whole number 30–100
    elif sensor_type == "motion":
        return random.choice([True, False])     # random True/False
    elif sensor_type == "door":
        return random.choice(["OPEN", "CLOSED"])
    elif sensor_type == "smoke":
        return random.choice(["CLEAR", "DETECTED"])
    else:
        return None


def process_reading(sensor, reading_value, system_mode):
    alerts = []
    timestamp = datetime.now().strftime("%H:%M:%S")
    sensor_type = sensor["sensor_type"]

    # 1. Security alerts (motion/door when AWAY or SLEEP)
    if should_trigger_security_alert(sensor, reading_value, system_mode):
        if sensor_type in ("motion", "door"):
            alerts.append(create_alert(
                "HIGH",
                f"Security: {sensor_type} activity at {sensor['location']}",
                sensor["sensor_id"], timestamp
            ))

    # 2. Smoke (its own thing, never overlaps temperature)
    if sensor_type == "smoke" and reading_value == "DETECTED":
        alerts.append(create_alert(
            "CRITICAL",
            f"Smoke detected in {sensor['location']}!",
            sensor["sensor_id"], timestamp
        ))

    # 3. Temperature: safety OR comfort, never both
    if sensor_type == "temperature":
        if is_abnormal_reading(sensor, reading_value):
            alerts.append(create_alert(
                "HIGH",
                f"Extreme temperature ({reading_value}°F) in {sensor['location']}",
                sensor["sensor_id"], timestamp
            ))
        elif system_mode == "HOME" and (reading_value < 60 or reading_value > 80):
            alerts.append(create_alert(
                "LOW",
                f"Temperature ({reading_value}°F) is a bit uncomfortable in {sensor['location']}",
                sensor["sensor_id"], timestamp
            ))

    return alerts

In [30]:
def trigger_alert(alert):
    """
    Displays an alert to the user.
    
    Parameters:
    - alert: Alert dictionary
    """
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")
 
def log_event(message, timestamp=None):
    """
    Logs an event to the console.
    
    Parameters:
    - message: The message to log
    - timestamp: Optional timestamp (uses current time if not provided)
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")

In [33]:
# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")
 
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
if alerts:
    trigger_alert(alerts[0])

Generated reading for Living Room: True
[ALERT!] 🚨 HIGH: Security: motion activity at Living Room


In [34]:
# Creatting classes

class Sensor:
    """
    Represents a sensor in the HomeGuard system.
    """

    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.id = sensor_id
        self.location = location
        self.sensor_type = sensor_type
        self.threshold = threshold
        self.current_value = None      # no reading taken yet

    def read(self):
        """Generates and stores a new reading for this sensor."""
        if self.sensor_type == "temperature":
            self.current_value = random.randint(30, 100)
        elif self.sensor_type == "motion":
            self.current_value = random.choice([True, False])
        elif self.sensor_type == "door":
            self.current_value = random.choice(["OPEN", "CLOSED"])
        elif self.sensor_type == "smoke":
            self.current_value = random.choice(["CLEAR", "DETECTED"])
        else:
            self.current_value = None
        return self.current_value

    def isAbnormal(self):
        """Checks if the current reading is abnormal."""
        if self.sensor_type == "temperature":
            return self.current_value < 35 or self.current_value > 95
        elif self.sensor_type == "motion":
            return self.current_value == True
        elif self.sensor_type == "door":
            return self.current_value == "OPEN"
        elif self.sensor_type == "smoke":
            return self.current_value == "DETECTED"
        else:
            return False

    def reset(self):
        """Resets the sensor's current reading to None."""
        self.current_value = None

    def __str__(self):
        status = "No reading" if self.current_value is None else str(self.current_value)
        return f"{self.id} ({self.location}): {status}"

In [36]:
# Create and test a sensor
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")

Sensor reading: 96
Is abnormal: True
Sensor info: TEST_001 (Test Room): 96


In [ ]:
# Integration of the Sensor class into a simulation

def run_simulation(duration_minutes=5, system_mode="AWAY"):
    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}\n")

    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]

    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")

        for sensor in sensors:
            reading = sensor.read()

            if sensor.sensor_type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.sensor_type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.sensor_type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.sensor_type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")

            # process_reading expects a dictionary, so we convert
            # the Sensor object into the dict shape it understands
            sensor_dict = {
                "sensor_id": sensor.id,
                "location": sensor.location,
                "sensor_type": sensor.sensor_type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)

            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")

        import time
        time.sleep(0.5)

    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)

In [ ]:
# test mode AWAY

run_simulation(duration_minutes=3, system_mode="AWAY")


=== HomeGuard Security System ===
Mode: AWAY


Time: 16:11:53
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 95°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: Security: door activity at Front Door
[LOG] [16:11:53] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 16:11:53
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 55°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 16:11:54
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Security: motion activity at Living Room
[LOG] [16:11:54] Sending notification to homeowner...
[READING] Kitchen Temperature: 76°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Simulation complete!


In [39]:
# Test mode HOME
run_simulation(duration_minutes=3, system_mode="HOME")

=== HomeGuard Security System ===
Mode: HOME


Time: 16:16:57
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 58°F (Abnormal)
[ALERT!] ℹ️ LOW: Temperature (58°F) is a bit uncomfortable in Kitchen
[READING] Front Door: OPEN
[READING] Bedroom Smoke: CLEAR

Time: 16:16:57
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 40°F (Abnormal)
[ALERT!] ℹ️ LOW: Temperature (40°F) is a bit uncomfortable in Kitchen
[READING] Front Door: OPEN
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🔥 CRITICAL: Smoke detected in Bedroom!
[LOG] [16:16:57] Sending notification to homeowner...

Time: 16:16:58
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 41°F (Abnormal)
[ALERT!] ℹ️ LOW: Temperature (41°F) is a bit uncomfortable in Kitchen
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Simulation complete!


In [41]:
# test mode SLEEP

run_simulation(duration_minutes=3, system_mode="SLEEP")


=== HomeGuard Security System ===
Mode: SLEEP


Time: 16:19:14
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Security: motion activity at Living Room
[LOG] [16:19:14] Sending notification to homeowner...
[READING] Kitchen Temperature: 75°F (Normal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: Security: door activity at Front Door
[LOG] [16:19:14] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 16:19:14
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 88°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: Security: door activity at Front Door
[LOG] [16:19:14] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 16:19:15
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 75°F (Normal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Simulation complete!
